# TabICLv2 context ladder (Google Colab, T4 GPU)

Runs the frozen TabICLv2 at the same nested contexts as the TabPFN-2.5 ladder (500, 1,000, 1,500, 2,000, 3,000, 4,000,
6,000, 8,000, 10,000 rows and the full training split) on the same three datasets and 10 seeds, and **saves its
predictions**. It also times TabICLv2 on this GPU and saves the model's architecture settings (a small JSON file), so
the CPU timing can be measured on the same machine as the TabPFN-2.5 CPU timing. All corrections then run on a CPU.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*. TabICLv2 needs no token or licence.
2. Upload `scan-repair-foundation-models.zip` (the code repository) to Google Drive, folder `accml_rerun`.

**Time on a T4:** about 2–3 hours. If Colab disconnects, reconnect and run all cells again: finished files are skipped.

**Send back:** `My Drive/accml_rerun/tabicl_ladder_results.zip` (made by the last cell).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: switch the runtime to a GPU'

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/accml_rerun'
assert os.path.exists(f'{DRIVE}/scan-repair-foundation-models.zip'), 'Upload scan-repair-foundation-models.zip into My Drive/accml_rerun first'
!rm -rf /content/work && mkdir -p /content/work && unzip -q "{DRIVE}/scan-repair-foundation-models.zip" -d /content/work
%cd /content/work/scan-repair-foundation-models
!pip install -q -r requirements.txt "tabicl==2.2.0"
OUT = f'{DRIVE}/tabicl_ladder_results'; os.makedirs(OUT, exist_ok=True)
os.environ['SSR_ARTIFACTS'] = OUT

**1. Context ladder** — 3 datasets × 10 seeds × up to 10 contexts (240 files). Progress lines look like `credit seed 42001 ctx 2000: fit … predict …`.

In [ ]:
%cd /content/work/scan-repair-foundation-models/experiments
!LADDER="500 1000 1500 2000 3000 4000 6000 8000 10000 full" OUT={OUT}/logits/tabicl_ladder python -u base_tabicl.py

**2. GPU timing** — seconds to score 1,000 rows at each context on this T4 (about 10 minutes).

In [ ]:
!OUT={OUT}/timing python -u timing_tabicl.py

**3. Architecture settings** for the CPU timing (no weights are saved, only the configuration).

In [ ]:
import json, numpy as np, pandas as pd
from tabicl import TabICLClassifier
m = TabICLClassifier(checkpoint_version='tabicl-classifier-v2-20260212.ckpt')
m.fit(pd.DataFrame(np.random.rand(50, 3)), np.r_[np.zeros(25), np.ones(25)].astype(int))
json.dump(m.model_config_, open(f'{OUT}/tabicl_v2_config.json', 'w'), indent=1, default=str)
print(json.dumps(m.model_config_, default=str)[:400])

In [ ]:
import glob
print(len(glob.glob(f'{OUT}/logits/tabicl_ladder/*.npz')), 'logits files (expected 240)')
!cd "{DRIVE}" && rm -f tabicl_ladder_results.zip && zip -qr tabicl_ladder_results.zip tabicl_ladder_results && ls -la "{DRIVE}/tabicl_ladder_results.zip"
print('Send back: My Drive/accml_rerun/tabicl_ladder_results.zip')